### Atividade Busca Semântica

Nesta atividade você deve aplicar os conhecimentos sobre word embeddings e tokenização para criar um mecanismos de busca semântica. Será disponibilizado um conjunto de dados que possui perguntas médicas, em inglês. Esse conjunto de dados possui as perguntas e também as respostas.

Você deve gerar os vetores das perguntas do conjunto de dados, e permitir que o "usuário" envie a sua pergunta. Você também deve gerar o vetor da pergunta do usuário e com isso buscar a resposta ideal para o usuário. **A resposta ideal é aquela onde o vetor da pergunta do usuário é mais similar ao vetor da pergunta do conjunto de dados** Consulte o notebook "nlp2.ipynb" para verificar como realizamos esse processo

Portanto, no seu script deve ser possível escrever um texto "pergunta" e deve ser retornado a resposta adequada, isto é, a resposta associada a pergunta mais similar no conjunto de dados.

Para isso utilize o pandas e os packages do huggingface

In [18]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers.tokenization_utils_base import BatchEncoding
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import kagglehub

# Download do conjunto de dados
print("Baixando dataset...")
try:
    path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")
    print("Path to dataset files:", path)
    
    # Carregamento do dataset
    df = pd.read_csv(f"{path}/medquad.csv")
    df_amostra = df.sample(5000, random_state=42)
    print(f"Dataset carregado com {len(df_amostra)} amostras")
    
except Exception as e:
    print(f"Erro ao baixar dataset: {e}")
    print("Criando dataset de exemplo...")
    
    # Dataset de exemplo para demonstração
    perguntas_exemplo = [
        "What are the symptoms of diabetes?",
        "How is high blood pressure treated?",
        "What causes migraine headaches?",
        "What are the risk factors for heart disease?",
        "How is asthma diagnosed?",
        "What is the treatment for depression?",
        "What are the symptoms of pneumonia?",
        "How can I prevent kidney stones?",
        "What causes back pain?",
        "How is arthritis treated?",
        "What are the signs of a heart attack?",
        "How to manage diabetes?",
        "What are the causes of hypertension?",
        "How to treat chronic pain?",
        "What are the symptoms of anxiety?"
    ]
    
    respostas_exemplo = [
        "Common symptoms of diabetes include increased thirst, frequent urination, extreme fatigue, and blurred vision.",
        "High blood pressure is typically treated with lifestyle changes and medications such as ACE inhibitors or diuretics.",
        "Migraine headaches can be caused by hormonal changes, certain foods, stress, and environmental factors.",
        "Risk factors for heart disease include high cholesterol, smoking, diabetes, obesity, and family history.",
        "Asthma is diagnosed through lung function tests, medical history, and physical examination of breathing patterns.",
        "Depression treatment typically involves psychotherapy, medications like antidepressants, and lifestyle changes.",
        "Pneumonia symptoms include cough with phlegm, fever, chills, and difficulty breathing.",
        "Kidney stones can be prevented by staying hydrated, reducing sodium intake, and limiting certain foods.",
        "Back pain can be caused by muscle strain, herniated discs, arthritis, or poor posture.",
        "Arthritis treatment includes medications, physical therapy, exercise, and sometimes surgery.",
        "Signs of heart attack include chest pain, shortness of breath, nausea, and pain in arms or jaw.",
        "Diabetes management involves blood sugar monitoring, proper diet, exercise, and medication as prescribed.",
        "Hypertension causes include genetics, poor diet, lack of exercise, stress, and excessive salt intake.",
        "Chronic pain treatment may include medications, physical therapy, relaxation techniques, and lifestyle changes.",
        "Anxiety symptoms include excessive worry, restlessness, fatigue, difficulty concentrating, and panic attacks."
    ]
    
    df_amostra = pd.DataFrame({
        'question': perguntas_exemplo,
        'answer': respostas_exemplo
    })

# Modelo para a língua inglesa
nome_modelo = "bert-base-uncased"
print(f"Carregando modelo: {nome_modelo}")
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
model = AutoModel.from_pretrained(nome_modelo)

## Funções para: 1) Obter os tokens; 2) Obter os embeddings
def get_tokens(pergunta: str) -> BatchEncoding:
    """
    Tokeniza uma pergunta usando o tokenizer do BERT
    """
    return tokenizer(pergunta, return_tensors="pt", truncation=True, padding=True, max_length=512)

def get_vetores(tokens_pergunta: BatchEncoding) -> Tensor:
    """
    Gera embeddings a partir dos tokens usando BERT
    """
    with torch.no_grad():
        outputs = model(**tokens_pergunta)
        # Usar mean pooling para obter um único vetor por pergunta
        embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

# Processamento do dataset
print("Processando dataset...")

## Novas colunas para os tokens e para os embeddings
df_amostra['tokens'] = df_amostra['question'].apply(lambda x: get_tokens(x))
df_amostra['vetores'] = df_amostra['tokens'].apply(lambda x: get_vetores(x))

# Converter embeddings para numpy array para facilitar cálculos
print("Convertendo vetores para numpy array...")
vetores_dataset = torch.stack([v.squeeze() for v in df_amostra['vetores']]).numpy()
print(f"Shape dos vetores do dataset: {vetores_dataset.shape}")

def buscar_resposta_similar(pergunta_usuario: str, top_k=1):
    """
    Busca a resposta mais similar à pergunta do usuário
    
    Args:
        pergunta_usuario: Pergunta feita pelo usuário
        top_k: Número de respostas similares a retornar
    
    Returns:
        Lista com as respostas mais similares
    """
    print(f"\n🔍 Buscando resposta para: '{pergunta_usuario}'")
    
    # Gerar embedding da pergunta do usuário
    tokens_usuario = get_tokens(pergunta_usuario)
    vetor_usuario = get_vetores(tokens_usuario).numpy()
    
    # Calcular similaridade de cosseno
    similaridades = cosine_similarity(vetor_usuario, vetores_dataset)[0]
    
    # Encontrar as perguntas mais similares
    indices_similares = np.argsort(similaridades)[::-1][:top_k]
    
    resultados = []
    for i, idx in enumerate(indices_similares):
        similaridade = similaridades[idx]
        pergunta_similar = df_amostra.iloc[idx]['question']
        resposta = df_amostra.iloc[idx]['answer']
        
        resultado = {
            'rank': i + 1,
            'similaridade': similaridade,
            'pergunta_similar': pergunta_similar,
            'resposta': resposta
        }
        resultados.append(resultado)
        
        print(f"\n✅ Resposta encontrada (Similaridade: {similaridade:.4f})")
        print(f"Pergunta similar: {pergunta_similar}")
        print(f"Resposta: {resposta}")
        print("-" * 80)
    
    return resultados

# Função principal para testar o sistema
def main():
    print("\n" + "="*80)
    print("🏥 SISTEMA DE BUSCA SEMÂNTICA MÉDICA")
    print("="*80)
    
    # Modo interativo
    print("\n" + "="*80)
    print("💬 MODO INTERATIVO")
    print("Digite sua pergunta médica em inglês (ou 'quit' para sair):")
    print("="*80)
    
    while True:
        pergunta_usuario = input("\n🤔 Sua pergunta: ").strip()
        
        if pergunta_usuario.lower() in ['quit', 'exit', 'sair', '']:
            print("👋 Obrigado por usar o sistema!")
            break
        
        try:
            buscar_resposta_similar(pergunta_usuario)
        except Exception as e:
            print(f"❌ Erro ao processar pergunta: {e}")

# Executar o sistema
if __name__ == "__main__":
    main()

Baixando dataset...
Path to dataset files: /home/codespace/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1
Dataset carregado com 5000 amostras
Carregando modelo: bert-base-uncased
Processando dataset...
Convertendo vetores para numpy array...
Shape dos vetores do dataset: (5000, 768)

🏥 SISTEMA DE BUSCA SEMÂNTICA MÉDICA

💬 MODO INTERATIVO
Digite sua pergunta médica em inglês (ou 'quit' para sair):

🔍 Buscando resposta para: 'diabetes'

✅ Resposta encontrada (Similaridade: 0.6206)
Pergunta similar: What causes Diabetes ?
Resposta: Type 1 diabetes is an autoimmune disease. In an autoimmune reaction, antibodies, or immune cells, attach to the body's own healthy tissues by mistake, signaling the body to attack them. At present, scientists do not know exactly what causes the body's immune system to attack the insulin-producing cells in the pancreas in people with type 1 diabetes. However, many believe that both genetic factors and environment